# Steel Defect Segmentation Pipeline

Pipeline segmentasi untuk deteksi defect pada steel menggunakan model ensemble U-Net dan FPN dengan EfficientNet-B3 encoder.

## 1. Import Libraries

In [1]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
import os
from sklearn.model_selection import train_test_split
from scipy.ndimage import label as scipy_label
import albumentations as A
from albumentations.pytorch import ToTensorV2

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
import segmentation_models_pytorch as smp

import warnings
warnings.filterwarnings('ignore')

ValueError: numpy.dtype size changed, may indicate binary incompatibility. Expected 96 from C header, got 88 from PyObject

## 2. Configuration

In [ ]:
DATA_DIR = Path('severstal-steel-defect-detection')
TRAIN_DIR = DATA_DIR / 'train_images'
TEST_DIR = DATA_DIR / 'test_images'
TRAIN_CSV = DATA_DIR / 'train.csv'
SAMPLE_SUB = DATA_DIR / 'sample_submission.csv'

CLASSIFICATION_RESULTS = Path('results-4/classification_results.csv')

IMG_HEIGHT = 256
IMG_WIDTH = 512
NUM_CLASSES = 4
BATCH_SIZE = 12
ACCUMULATION_STEPS = 2
NUM_EPOCHS = 30
LEARNING_RATE = 1e-4
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

LABEL_THRESHOLDS = [0.7, 0.7, 0.6, 0.6]
PIXEL_THRESHOLDS = [0.55, 0.55, 0.55, 0.55]
MIN_PIXELS = [600, 600, 900, 2000]
MIN_COMPONENT_SIZE = 150

print(f'Using device: {DEVICE}')

## 3. RLE Encoding/Decoding Utilities

In [ ]:
def rle_to_mask(rle_string, height=256, width=1600):
    if pd.isna(rle_string) or rle_string == '':
        return np.zeros((height, width), dtype=np.uint8)
    
    s = rle_string.split()
    starts, lengths = [np.asarray(x, dtype=int) for x in (s[0::2], s[1::2])]
    starts -= 1
    ends = starts + lengths
    img = np.zeros(height * width, dtype=np.uint8)
    for lo, hi in zip(starts, ends):
        img[lo:hi] = 1
    return img.reshape((height, width), order='F')

def mask_to_rle(mask):
    pixels = mask.T.flatten()
    pixels = np.concatenate([[0], pixels, [0]])
    runs = np.where(pixels[1:] != pixels[:-1])[0] + 1
    runs[1::2] -= runs[::2]
    return ' '.join(str(x) for x in runs)

def build_masks(df, image_id, height=256, width=1600):
    masks = np.zeros((height, width, NUM_CLASSES), dtype=np.uint8)
    for class_id in range(1, NUM_CLASSES + 1):
        mask_data = df[(df['ImageId'] == image_id) & (df['ClassId'] == class_id)]
        if not mask_data.empty:
            rle = mask_data.iloc[0]['EncodedPixels']
            masks[:, :, class_id - 1] = rle_to_mask(rle, height, width)
    return masks

## 4. Load Data

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
sample_sub = pd.read_csv(SAMPLE_SUB)
classification_df = pd.read_csv(CLASSIFICATION_RESULTS)

print(f'Training data shape: {train_df.shape}')
print(f'Unique images: {train_df["ImageId"].nunique()}')

images_with_defects = classification_df[classification_df['HasDefect'] == 1]['ImageId'].unique()
print(f'\nImages with predicted defects (from classification): {len(images_with_defects)}')

## 5. Dataset Class

In [ ]:
class SteelSegmentationDataset(Dataset):
    def __init__(self, df, image_ids, img_dir, transform=None, is_test=False, tta_mode=None):
        self.df = df
        self.image_ids = image_ids
        self.img_dir = img_dir
        self.transform = transform
        self.is_test = is_test
        self.tta_mode = tta_mode
    
    def __len__(self):
        return len(self.image_ids)
    
    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_path = self.img_dir / image_id
        
        image = cv2.imread(str(img_path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.is_test:
            if self.tta_mode == 'hflip':
                image = cv2.flip(image, 1)
            elif self.tta_mode == 'vflip':
                image = cv2.flip(image, 0)
            
            if self.transform:
                augmented = self.transform(image=image)
                image = augmented['image']
            return image, image_id
        
        masks = build_masks(self.df, image_id, image.shape[0], image.shape[1])
        
        if self.transform:
            augmented = self.transform(image=image, mask=masks)
            image = augmented['image']
            masks = augmented['mask']
            
            if isinstance(masks, torch.Tensor):
                masks = masks.numpy()
            
            masks = np.transpose(masks, (2, 0, 1)).astype(np.float32)
        else:
            masks = masks.transpose(2, 0, 1).astype(np.float32)
        
        return image, torch.FloatTensor(masks)

## 6. Data Augmentation

In [ ]:
train_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomBrightnessContrast(p=0.3),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(IMG_HEIGHT, IMG_WIDTH),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

## 7. Prepare Data Splits

In [ ]:
all_image_ids = train_df['ImageId'].unique()
train_ids, val_ids = train_test_split(all_image_ids, test_size=0.2, random_state=42)

print(f'Training samples: {len(train_ids)}')
print(f'Validation samples: {len(val_ids)}')

## 8. Loss Functions

In [ ]:
class BCEWithPosWeightLoss(nn.Module):
    def __init__(self, pos_weight):
        super(BCEWithPosWeightLoss, self).__init__()
        self.pos_weight = torch.tensor(pos_weight, dtype=torch.float32)
    
    def forward(self, logits, targets):
        # Move pos_weight to the same device as logits
        pos_weight = self.pos_weight.to(logits.device)
        
        # Reshape pos_weight to match the number of classes
        # logits shape: [batch_size, num_classes, height, width]
        # pos_weight shape should be: [num_classes] or [1, num_classes, 1, 1]
        if pos_weight.dim() == 1:
            pos_weight = pos_weight.view(1, -1, 1, 1)
        
        loss = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=pos_weight
        )
        return loss

class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super(DiceLoss, self).__init__()
        self.smooth = smooth
    
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        probs_flat = probs.view(-1)
        targets_flat = targets.view(-1)
        
        intersection = (probs_flat * targets_flat).sum()
        dice = (2. * intersection + self.smooth) / (probs_flat.sum() + targets_flat.sum() + self.smooth)
        
        return 1 - dice

class BCEDiceLoss(nn.Module):
    def __init__(self, pos_weight, bce_weight=0.75, dice_weight=0.25):
        super(BCEDiceLoss, self).__init__()
        self.bce = BCEWithPosWeightLoss(pos_weight)
        self.dice = DiceLoss()
        self.bce_weight = bce_weight
        self.dice_weight = dice_weight
    
    def forward(self, logits, targets):
        return self.bce_weight * self.bce(logits, targets) + self.dice_weight * self.dice(logits, targets)

def dice_coefficient(pred, target, threshold=0.5):
    pred = (pred > threshold).float()
    intersection = (pred * target).sum()
    return (2. * intersection) / (pred.sum() + target.sum() + 1e-8)

## 9. Training Functions

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device, scaler, accumulation_steps):
    model.train()
    total_loss = 0
    total_dice = 0
    
    optimizer.zero_grad()
    
    for i, (images, masks) in enumerate(loader):
        images = images.to(device)
        masks = masks.to(device)
        
        with autocast():
            outputs = model(images)
            loss = criterion(outputs, masks) / accumulation_steps
        
        scaler.scale(loss).backward()
        
        if (i + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        
        total_loss += loss.item() * accumulation_steps
        
        with torch.no_grad():
            dice = dice_coefficient(torch.sigmoid(outputs), masks)
            total_dice += dice.item()
    
    return total_loss / len(loader), total_dice / len(loader)

def validate_epoch(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    total_dice = 0
    
    with torch.no_grad():
        for images, masks in loader:
            images = images.to(device)
            masks = masks.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, masks)
            
            total_loss += loss.item()
            
            dice = dice_coefficient(torch.sigmoid(outputs), masks)
            total_dice += dice.item()
    
    return total_loss / len(loader), total_dice / len(loader)

def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, num_epochs, model_path):
    scaler = GradScaler()
    best_dice = 0
    history = {'train_loss': [], 'train_dice': [], 'val_loss': [], 'val_dice': []}
    
    for epoch in range(num_epochs):
        train_loss, train_dice = train_epoch(
            model, train_loader, criterion, optimizer, DEVICE, scaler, ACCUMULATION_STEPS
        )
        val_loss, val_dice = validate_epoch(model, val_loader, criterion, DEVICE)
        
        history['train_loss'].append(train_loss)
        history['train_dice'].append(train_dice)
        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_dice)
        
        scheduler.step()
        
        print(f'Epoch {epoch+1}/{num_epochs}')
        print(f'Train Loss: {train_loss:.4f}, Train Dice: {train_dice:.4f}')
        print(f'Val Loss: {val_loss:.4f}, Val Dice: {val_dice:.4f}')
        
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), model_path)
            print(f'Model saved with Dice: {best_dice:.4f}')
        print('-' * 50)
    
    return history

## 10. Train U-Net with BCE Loss

In [ ]:
train_dataset = SteelSegmentationDataset(train_df, train_ids, TRAIN_DIR, transform=train_transform)
val_dataset = SteelSegmentationDataset(train_df, val_ids, TRAIN_DIR, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print('Training U-Net with EfficientNet-B3 encoder and BCE loss...')
unet_bce = smp.Unet(
    encoder_name='efficientnet-b3',
    encoder_weights='imagenet',
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None
).to(DEVICE)

criterion_bce = BCEWithPosWeightLoss(pos_weight=[2.0, 2.0, 1.0, 1.5])
optimizer = torch.optim.RAdam(unet_bce.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

history_unet_bce = train_model(
    unet_bce, train_loader, val_loader, criterion_bce, 
    optimizer, scheduler, NUM_EPOCHS, 'unet_efficientb3_bce.pth'
)

## 11. Train FPN Models

In [ ]:
print('Training FPN models with EfficientNet-B3 encoder...')

for i in range(1, 4):
    print(f'\nTraining FPN model {i}/3 with BCE loss...')
    fpn = smp.FPN(
        encoder_name='efficientnet-b3',
        encoder_weights='imagenet',
        in_channels=3,
        classes=NUM_CLASSES,
        activation=None
    ).to(DEVICE)
    
    criterion_bce = BCEWithPosWeightLoss(pos_weight=[2.0, 2.0, 1.0, 1.5])
    optimizer = torch.optim.RAdam(fpn.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    
    history = train_model(
        fpn, train_loader, val_loader, criterion_bce, 
        optimizer, scheduler, NUM_EPOCHS, f'fpn_efficientb3_bce_{i}.pth'
    )
    
    print(f'\nFine-tuning FPN model {i}/3 with BCE+Dice loss...')
    fpn.load_state_dict(torch.load(f'fpn_efficientb3_bce_{i}.pth'))
    
    criterion_bcedice = BCEDiceLoss(pos_weight=[2.0, 2.0, 1.0, 1.5])
    optimizer = torch.optim.RAdam(fpn.parameters(), lr=LEARNING_RATE / 10)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
    
    history_ft = train_model(
        fpn, train_loader, val_loader, criterion_bcedice, 
        optimizer, scheduler, 10, f'fpn_efficientb3_bcedice_{i}.pth'
    )

## 12. Train Additional FPN Models with BCE

In [ ]:
for i in range(1, 3):
    print(f'\nTraining additional FPN model {i}/2 with BCE loss...')
    fpn_bce = smp.FPN(
        encoder_name='efficientnet-b3',
        encoder_weights='imagenet',
        in_channels=3,
        classes=NUM_CLASSES,
        activation=None
    ).to(DEVICE)
    
    criterion_bce = BCEWithPosWeightLoss(pos_weight=[2.0, 2.0, 1.0, 1.5])
    optimizer = torch.optim.RAdam(fpn_bce.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)
    
    history = train_model(
        fpn_bce, train_loader, val_loader, criterion_bce, 
        optimizer, scheduler, NUM_EPOCHS, f'fpn_efficientb3_bce_extra_{i}.pth'
    )

## 13. Postprocessing Functions

In [ ]:
def remove_small_components(mask, min_size=150):
    labeled, num_features = scipy_label(mask)
    
    for i in range(1, num_features + 1):
        component = (labeled == i)
        if component.sum() < min_size:
            mask[component] = 0
    
    return mask

def postprocess_mask(mask, class_id, pixel_threshold, min_pixels, min_component_size):
    mask = (mask > pixel_threshold).astype(np.uint8)
    
    if mask.sum() < min_pixels:
        return np.zeros_like(mask)
    
    mask = remove_small_components(mask, min_component_size)
    
    return mask

## 14. Prediction with TTA and Ensemble

In [ ]:
def predict_with_tta(model, image, device, tta_mode=None):
    model.eval()
    
    if tta_mode == 'hflip':
        image_tta = cv2.flip(image, 1)
    elif tta_mode == 'vflip':
        image_tta = cv2.flip(image, 0)
    else:
        image_tta = image.copy()
    
    augmented = val_transform(image=image_tta)
    image_tensor = augmented['image'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = torch.sigmoid(model(image_tensor))
        pred = output.cpu().numpy()[0]
    
    if tta_mode == 'hflip':
        pred = np.flip(pred, axis=2)
    elif tta_mode == 'vflip':
        pred = np.flip(pred, axis=1)
    
    return pred

def get_ensemble_predictions(models, image, device, tta_modes=['none', 'hflip', 'vflip']):
    all_preds = []
    
    for model in models:
        for tta_mode in tta_modes:
            tta = tta_mode if tta_mode != 'none' else None
            pred = predict_with_tta(model, image, device, tta)
            all_preds.append(pred)
    
    ensemble_pred = np.mean(all_preds, axis=0)
    return ensemble_pred

## 15. Load Trained Models for Ensemble

In [ ]:
ensemble_models = []

print('Loading U-Net BCE model...')
unet_bce_model = smp.Unet(
    encoder_name='efficientnet-b3',
    encoder_weights='imagenet',
    in_channels=3,
    classes=NUM_CLASSES,
    activation=None
).to(DEVICE)
unet_bce_model.load_state_dict(torch.load('unet_efficientb3_bce.pth'))
ensemble_models.append(unet_bce_model)

print('Loading FPN BCE+Dice models...')
for i in range(1, 4):
    fpn_model = smp.FPN(
        encoder_name='efficientnet-b3',
        encoder_weights='imagenet',
        in_channels=3,
        classes=NUM_CLASSES,
        activation=None
    ).to(DEVICE)
    fpn_model.load_state_dict(torch.load(f'fpn_efficientb3_bcedice_{i}.pth'))
    ensemble_models.append(fpn_model)

print('Loading additional FPN BCE models...')
for i in range(1, 3):
    fpn_model = smp.FPN(
        encoder_name='efficientnet-b3',
        encoder_weights='imagenet',
        in_channels=3,
        classes=NUM_CLASSES,
        activation=None
    ).to(DEVICE)
    fpn_model.load_state_dict(torch.load(f'fpn_efficientb3_bce_extra_{i}.pth'))
    ensemble_models.append(fpn_model)

print(f'Total models in ensemble: {len(ensemble_models)}')

## 16. Generate Predictions for Test Set

In [ ]:
test_images = sorted([f.name for f in TEST_DIR.glob('*.jpg')])

classification_pred = {}
for _, row in classification_df.iterrows():
    if row['ImageId'] not in classification_pred:
        classification_pred[row['ImageId']] = {}
    classification_pred[row['ImageId']][row['ClassId']] = row['HasDefect']

print(f'Processing {len(test_images)} test images...')

submission_data = []

for idx, image_id in enumerate(test_images):
    if (idx + 1) % 100 == 0:
        print(f'Processed {idx + 1}/{len(test_images)} images')
    
    img_path = TEST_DIR / image_id
    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    original_height, original_width = image.shape[:2]
    
    has_any_defect = False
    if image_id in classification_pred:
        for class_id in range(1, NUM_CLASSES + 1):
            if classification_pred[image_id].get(class_id, 0) == 1:
                has_any_defect = True
                break
    
    if not has_any_defect:
        for class_id in range(1, NUM_CLASSES + 1):
            submission_data.append({
                'ImageId_ClassId': f'{image_id}_{class_id}',
                'EncodedPixels': ''
            })
        continue
    
    ensemble_pred = get_ensemble_predictions(ensemble_models, image, DEVICE)
    
    for class_id in range(NUM_CLASSES):
        pred_mask = ensemble_pred[class_id]
        
        pred_mask_resized = cv2.resize(
            pred_mask, 
            (original_width, original_height), 
            interpolation=cv2.INTER_LINEAR
        )
        
        label_threshold = LABEL_THRESHOLDS[class_id]
        if image_id in classification_pred and classification_pred[image_id].get(class_id + 1, 0) < label_threshold:
            rle = ''
        else:
            pred_mask_processed = postprocess_mask(
                pred_mask_resized,
                class_id,
                PIXEL_THRESHOLDS[class_id],
                MIN_PIXELS[class_id],
                MIN_COMPONENT_SIZE
            )
            
            if pred_mask_processed.sum() > 0:
                rle = mask_to_rle(pred_mask_processed)
            else:
                rle = ''
        
        submission_data.append({
            'ImageId_ClassId': f'{image_id}_{class_id + 1}',
            'EncodedPixels': rle
        })

print('Predictions complete!')

## 17. Create Submission File

In [ ]:
submission_df = pd.DataFrame(submission_data)
submission_df.to_csv('submission.csv', index=False)

print(f'Submission file created with {len(submission_df)} rows')
print(f'\nSubmission summary:')
print(f'Total predictions: {len(submission_df)}')

masks_with_defects = submission_df[submission_df['EncodedPixels'] != ''].shape[0]
masks_without_defects = submission_df[submission_df['EncodedPixels'] == ''].shape[0]

print(f'Masks with defects: {masks_with_defects} ({masks_with_defects/len(submission_df)*100:.1f}%)')
print(f'Masks without defects: {masks_without_defects} ({masks_without_defects/len(submission_df)*100:.1f}%)')

print('\nSample submission:')
print(submission_df.head(20))

## 18. Visualization

In [ ]:
sample_images = np.random.choice(images_with_defects, min(4, len(images_with_defects)), replace=False)

fig, axes = plt.subplots(len(sample_images), 5, figsize=(20, 4 * len(sample_images)))
if len(sample_images) == 1:
    axes = axes.reshape(1, -1)

for i, image_id in enumerate(sample_images):
    img_path = TEST_DIR / image_id
    image = cv2.imread(str(img_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    ensemble_pred = get_ensemble_predictions(ensemble_models, image, DEVICE)
    
    axes[i, 0].imshow(image)
    axes[i, 0].set_title(f'{image_id}')
    axes[i, 0].axis('off')
    
    for class_id in range(NUM_CLASSES):
        pred_mask = ensemble_pred[class_id]
        pred_mask_resized = cv2.resize(pred_mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_LINEAR)
        pred_mask_processed = postprocess_mask(
            pred_mask_resized,
            class_id,
            PIXEL_THRESHOLDS[class_id],
            MIN_PIXELS[class_id],
            MIN_COMPONENT_SIZE
        )
        
        axes[i, class_id + 1].imshow(pred_mask_processed, cmap='gray')
        axes[i, class_id + 1].set_title(f'Class {class_id + 1}')
        axes[i, class_id + 1].axis('off')

plt.tight_layout()
plt.show()